In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit, prange
%matplotlib inline

from scipy.spatial.distance import cdist
from random import sample
from itertools import product


In [ ]:
@njit #to adjust indexing
def compute_strides(n_cells):
    return np.array([1, n_cells[0], n_cells[0] * n_cells[1]], dtype=np.int32)

@njit
def compute_cell_sizes(lbox, rcut):
    dim = len(lbox)
    n_cells = np.empty_like(lbox, dtype=np.int32)
    cell_size = np.empty_like(lbox)
    for d in range(dim):
        n_cells[d] = int(lbox[d] / rcut)
        cell_size[d] = lbox[d] / n_cells[d]
    return n_cells, cell_size

@njit
def setup_linked_list(r, lbox, rcut, padding=0):
    N, dim = r.shape
    n_cells, cell_size = compute_cell_sizes(lbox, rcut+padding)
    strides = compute_strides(n_cells)
    cells = np.prod(n_cells)
    
    head = -np.ones(cells, dtype=np.int32)
    linked_list = -np.ones(N, dtype=np.int32) # works up to 2mil lol

    for i in range(N):
        r_cell = np.empty(dim, dtype=np.int32)
        for d in range(dim):
            r_cell[d] = int(r[i,d] / cell_size[d]) % n_cells[d]
        
        flat_index = np.sum(r_cell * strides)  # Faster with precomputed strides
        
        linked_list[i] = head[flat_index]
        head[flat_index] = i
    
    return head, linked_list, n_cells, cell_size, strides
    
# planned to dynamically update, but skipped due to time

# get_neighbors written by DeepSeek
@njit
def get_neighbors(i, r, head, linked_list, n_cells, cell_size, strides, PBC):
    """Faster neighbor search with precomputed strides"""
    neighbors = []
    ncx, ncy, ncz = n_cells
    cx = int(r[i,0] / cell_size[0]) % ncx
    cy = int(r[i,1] / cell_size[1]) % ncy
    cz = int(r[i,2] / cell_size[2]) % ncz
    
    for dz in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if PBC:
                    nx = (cx + dx) % ncx
                    ny = (cy + dy) % ncy
                    nz = (cz + dz) % ncz
                else:
                    nx, ny, nz = cx + dx, cy + dy, cz + dz
                    if (nx < 0 or nx >= ncx or 
                        ny < 0 or ny >= ncy or 
                        nz < 0 or nz >= ncz):
                        continue
                
                # Faster flat index calculation with strides
                flat_index = nx*strides[0] + ny*strides[1] + nz*strides[2]
                
                j = head[flat_index]
                while j != -1:
                    if j > i:
                        neighbors.append(j)
                    j = linked_list[j]
    return neighbors

In [ ]:
@njit(parallel=True)
def forces_lj( lbox, r, f, rcut, PBC, padding=0):

    # set the parameters for calculation
    N, nd = r.shape #read off the dimensions
    rcut_sq = rcut*rcut # to speed up comparisons #whats this for?
    dr = np.zeros(nd) # vector between two particles
    df = np.zeros(nd) # force contribution from one pair
    lbox2 = lbox/2 # speed up calculations
    
    # set the force array to zero
    f[:,:] = 0

    head, linked_list, n_cells, cell_size, strides = setup_linked_list(r, lbox, rcut, padding=padding)

    # loop over the first particle
    for i in prange(N):
        # loop over the second particle
        neighbors = get_neighbors(i, r, head, linked_list, n_cells, cell_size, strides, PBC)
        for j in neighbors:
            Dx = r[j] - r[i]
            if PBC:
                Dx = (Dx + lbox2) % lbox - lbox2

            # if dist_sq is less than rcut_sq, there is a contribution to the force from this pair
            dist_sq = np.sum(Dx**2)
            if dist_sq < rcut_sq:
                continue
            
            #df = 24/dist**8 * ( 1 - 2/dist) * Dx
            # much faster to pre-compute inverse distance
            # saves 4x time!!!!!
            inv_dist_sq = 1.0/dist_sq
            inv_d6 = inv_dist_sq**3
            inv_d12 = inv_d6**2

            df = (48*inv_d12-24*inv_d6)*inv_dist_sq*Dx


            f[i] += df
            f[j] -= df


def measure_kin( v, masses ):

    v2 = np.sum(v**2)
    return sum(v2*masses/2)

In [ ]:
# initialize particle positions
# Input:
# lbox -- array with sizes
# r -- allocated array of shape N * dim,
#      where N is the number of particles and dim is the dimensionality
# Output:
# the input r is changed
def initial_r( lbox, r ):
    N, dim = r.shape
    r[:] = np.random.rand(*r.shape) * lbox
    i = 1
    breaker = 0  

    while i < N:
        while (cdist(r[:N+1],r[:N+1]) + np.eye(len(r[:N+1]), len(r[:N+1]))*2 < 1).any():
            r[i] = np.random.rand(2) * lbox
            breaker += 1
            if breaker == 1000:
                r[:] = np.random.rand(*r.shape) * lbox
                breaker = 0
                i = 1
        breaker = 0
        i += 1
    
def easy_initial_r( lbox, r, min_dist = 1, jitter=0):
    grid_spacing = min_dist+jitter
    grids = np.meshgrid(*[np.arange(min_dist, l-min_dist, grid_spacing) for l in lbox])
    all_points = np.column_stack([grid.ravel() for grid in grids])

    apply_jitter = lambda x: x + (np.random.rand()-0.5)*jitter
    all_points = np.vectorize(apply_jitter)(all_points)
    #assert (cdist(all_points, all_points) + 2*min_dist*np.eye(len(all_points)) >= min_dist).all()
    r[:] = np.array(sample(list(all_points), k=len(r)))

def initial_v( lbox, v, T, masses ):
    masses = np.array([[m] for m in masses])
    sigma = np.sqrt(T/masses)

    v[:] = np.random.randn(*v.shape)  * sigma
    
    total_momentum = np.sum(masses * v, axis=1, keepdims=True)
    total_mass = np.sum(masses)
    
    v[:] -= total_momentum / total_mass

In [ ]:
def update_r( dt, r, v ):

    r[:] = r + v*dt

def update_v( dt, v, f, masses ):

    v[:] = v + f/np.array([[m] for m in masses])*dt

def boundary_periodic( lbox, r, v ):

    r[:] = r % lbox

def boundary_wall( lbox, r, v ):

    # loop on particles and r components,
    # if r is outside, reflect back in,
    # reverse the velocity if needed
    
    #safety
    if (abs(r - lbox/2) > lbox*1.5).any():
        raise ValueError('full ring not implemented!')

    #hard wall
    v[r<0] *= -1
    r[r<0] *= -1

    v[r>lbox] *= -1
    r[:] = np.where(r>lbox, 2*lbox-r, r)

boundary = [boundary_wall, boundary_periodic]


In [ ]:
### main parameters

# dimension
dim = 3
N = 10000
radius = (2)**(1/6)
rcut = 2.2 # field limits

# to simplify, box size should be >= 2*rcut
lbox = np.array( [15.0,15.0,15.0] )

# allocate arrays
r = np.zeros( (N,dim) )
v = np.zeros( (N,dim) )
f = np.zeros( (N,dim) )
masses = np.full( N, 1 )
Temp = 0.2
PBC = True

sim_fps = 30 
skiptime = 0
runtime = 20  # seconds
dt = 1/sim_fps

Nsim = int(runtime*sim_fps)
skipframes = int(skiptime*sim_fps)

Ninner = 1 # do not change!
frames = Nsim

# plot colors with force or velocity magnitude
velocity_colors = False

In [ ]:
from time import perf_counter

r_storage = np.zeros([Nsim,*r.shape])
v_storage = np.zeros([Nsim,*r.shape])
f_storage = np.zeros([Nsim,*f.shape])
kin_storage = []
pot_storage = []

easy_initial_r( lbox, r, min_dist=radius/4 )
initial_v( lbox, v, Temp, masses)

segment_times = np.zeros((Nsim, 5))

frame = 0
for i in range(Nsim+skipframes):
    # inner loop
    for j in range(Ninner):
        t0 = perf_counter()
        # propagate velocities by dt/2
        update_v(dt/2, v, f, masses)
        v *= 0.999
        t1 = perf_counter()
        # propagate positions r by dt
        update_r(dt, r, v)
        t2 = perf_counter()
        # apply boundary conditions
        boundary[PBC]( lbox, r, v )
        t3 = perf_counter()
        # compute forces
        forces_lj( lbox, r, f, rcut, PBC, padding = 0.1*rcut)
        t4 = perf_counter()

        # propagate velocities by dt/2
        update_v(dt/2, v, f, masses)
        v *= 0.999
        t5 = perf_counter()
    
    if i >= skipframes:
        r_storage[frame] = r
        v_storage[frame] = v
        f_storage[frame] = f
        kin_storage.append(measure_kin(v, masses))
        frame += 1

        segment_times[i-skipframes] = [t1-t0,t2-t1,t3-t2,t4-t3,t5-t4]
        
#    pot_storage.append(measure_pot(lbox, r, rcut, PBC)) #slow

# total_energy = np.array(kin_storage) + np.array(pot_storage)
# temperature = 2 * np.array(kin_storage) / (dim * N)

In [ ]:
framelist = [i for i in range(Nsim)]

names = ['1. v+at/2', '2. x+vt/2', '3. apply boundary', '4. forces', '5. v+at/2']

def trailing_mean(array, trail_len=None):
    ar2 = np.zeros(array.shape)
    if trail_len:
        for i in range(len(array)):
            ar2[i] = array[max(i-trail_len,0):i+1].mean()
    else:
        for i in range(len(array)):
            ar2[i] = array[:i+1].mean()
    return ar2

for i in range(5):
    plt.plot(range(len(segment_times)), trailing_mean(segment_times[:,i], 5), label=names[i])
plt.grid()
plt.yscale('log')
plt.legend()

[
(100, 0.0021920887780531, 7.725077775022428e-05), 
(200, 0.003957629223522316, 0.00015721183250813435), 
(300, 0.008905130444489057, 0.00022149038915004993), 
(400, 0.015655170388312805, 0.00027833994555597505), 
(500, 0.02454047416595535, 0.00034480144518763865), 
(600, 0.03576956905615387, 0.00039061083341948685), 
(700, 0.04890139611211554, 0.0004560316105036893), 
(800, 0.06384927600030399, 0.0006255316660260886)
]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
bounding = 0.5

plt.title(f'N={N} particles in LJ Fluid')

if PBC:
    ax.set_xlim(0+bounding, lbox[0]-bounding)
    ax.set_ylim(0+bounding, lbox[1]-bounding)
    ax.set_zlim(0+bounding, lbox[2]-bounding)
else:
    ax.set_xlim(0, lbox[0])
    ax.set_ylim(0, lbox[1])
    ax.set_zlim(0, lbox[2])


_storage = v_storage if velocity_colors else f_storage

max_norm = np.max(np.linalg.norm(_storage, axis=2)[1:])
_vec = _storage[-1]
_norms = np.linalg.norm(_vec, axis=1)  # Correct axis for norm calculation
initial_colors = _norms / max_norm

scat = ax.scatter(r_storage[-1, :, 0], r_storage[-1, :, 1], r_storage[-1, :, 2],
    s=radius**2/(lbox[0]*lbox[1]*lbox[2])*4e5/100,
    c=initial_colors,
    cmap='gnuplot2', vmin=0, vmax=1)

cbar = fig.colorbar(scat, ax=ax, shrink=0.6, aspect=20, label='Velocity Magnitude' if velocity_colors else 'Force Magnitude')

#quiver = ax.quiver(r_storage[0, :, 0], r_storage[0, :, 1], r_storage[0, :, 2],
#    f_storage[0, :, 0], f_storage[0, :, 1], f_storage[0, :, 2],
#    length=0.2, normalize=True)

def animate(ax, i):
    scat._offsets3d = (r_storage[i, :, 0], r_storage[i, :, 1], r_storage[i, :, 2])

    vecs = _storage[i]
    _norms = np.linalg.norm(vecs, axis=1)  # Correct axis for norm calculation
    _norms_normalized = _norms / max_norm

    scat.set_array(_norms_normalized) # updates the color per point
#    quiver.set_segments(np.array([r_storage[i], r_storage[i] + forces/np.linalg.norm(forces, axis=1, keepdims=True)*0.2 ]).transpose(1, 0, 2))

    return scat#, quiver

In [ ]:
from animator import ParticleAnimator
from matplotlib.animation import FFMpegWriter, PillowWriter
filename = f'{N}particles{runtime}sec.mp4'
display_fps = 30


anim = ParticleAnimator(fig, FFMpegWriter, filename)
anim.set_display(display_fps, runtime)
frame_data = anim.generate_frame_data([i for i in range(len(r_storage))], update_fps=30)
anim.animate_ax(ax, animate, frame_data)
anim.make_animation()
anim.display()